In [ ]:

import matplotlib.pyplot as plt
import time
import torch
import numpy as np
from sympy.codegen.rewriting import Optimization
from tqdm import tqdm
from tqdm.notebook import tqdm
import pandas as pd
import torch
import os
from datetime import datetime
import sys
import json
from matplotlib.ticker import LogLocator, LogFormatter
from tqdm import trange
from functools import partial

In [ ]:
!pip install flow_matching -q
!pip install POT -q

In [ ]:
import os
import sys
from huggingface_hub import login
from google.colab import userdata
import wandb

hf_token = userdata.get('HF')
if hf_token:
    login(token=hf_token)
else:
    login()

if 'google.colab' in str(get_ipython()):
    import getpass

    !pip install -q diffusers transformers accelerate xformers
    !pip install -q scikit-learn matplotlib Pillow

    github_token = userdata.get('GITHUB')
    if github_token:
        token = github_token
    else:
        token = getpass.getpass("Enter your GitHub personal access token: ")

    repo_url  = f"https://{token}@github.com/orineo1/conditional-matching-paper.git"
    repo_name = "conditional-matching-paper"
    branch    = "adding-simu-compare"

    if not os.path.exists(repo_name):
        !git clone {repo_url}
    else:
        print(f"✅ Repo '{repo_name}' already cloned — pulling latest...")
        !cd {repo_name} && git pull

    !cd {repo_name} && git checkout {branch}

    repo_path = f"/content/{repo_name}"
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

    print(f"\n✅ Repo ready. Branch: {branch}")
    print(f"📁 Python path: {repo_path}")

repo_path = f"/content/{repo_name}/simulations"
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)



In [ ]:
import importlib
import Diffusion
import superposition_utils

import LossFunctions
import ConsistencyModels
import  FlowMatching
from ConsistencyModels import ConsistencyModel,ConsistencyModeliCT
import dist_utils
import Optimization
import EncoderDecoder
import evalModels

importlib.reload(Diffusion)
importlib.reload(LossFunctions)

importlib.reload(ConsistencyModels)
importlib.reload(FlowMatching)
importlib.reload(dist_utils)
importlib.reload(Optimization)
importlib.reload(superposition_utils)
importlib.reload(EncoderDecoder)
importlib.reload(evalModels)



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2D cond on 1D

In [ ]:
mu_list = [torch.tensor([-5,5],dtype=torch.float64),
           torch.tensor([-5,-5],dtype=torch.float64),
           torch.tensor([5, 3],dtype=torch.float64),
           torch.tensor([5,-1],dtype=torch.float64),
           torch.tensor([0, -3],dtype=torch.float64),
           torch.tensor([-2,4 ],dtype=torch.float64),
           torch.tensor([-2,-3 ],dtype=torch.float64),
           torch.tensor([ 1,2],dtype=torch.float64),
           torch.tensor([-7,1],dtype=torch.float64),
           torch.tensor([7,5],dtype=torch.float64),
           torch.tensor([0,-5],dtype=torch.float64)
           ]

Sigma_list = [
    torch.tensor([[0.5000, 0.1950],
                  [0.1950, 0.2000]], dtype=torch.float64)
              ] * len(mu_list)

# Mixture weights
alpha =torch.tensor( [1 / len(mu_list)] * len(mu_list),dtype=torch.float64)

mu_list = [mu.float() for mu in mu_list]
Sigma_list = [cov.float() for cov in Sigma_list]
alpha = alpha.float()

## Target
x_star=torch.tensor([-5])
mu_temp, Sigma_temp =dist_utils.compute_conditionals(mu_list, Sigma_list, x_star)
temp_alpha = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_star)
mog_means, mog_variances, weights= dist_utils.filter_and_normalize(mu_temp, Sigma_temp, temp_alpha, threshold=0.01)


In [ ]:
X = dist_utils.generate_mog_samples(25_000, mu_list, Sigma_list, alpha).float().to(device)
xh_cpu = X.detach().cpu().numpy()

# Scatter plot of first column vs. second column
plt.scatter(xh_cpu[:, 0], xh_cpu[:, 1], alpha=0.6, s=20)
plt.title("Scatter Plot of P(X,Y)")
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)
plt.show()

## Parameters for tests

In [ ]:
## NN
nblocks=3
nunits=128
nepochs=20_000
batch_size=512

nepochs_CM=7_500
batch_size_CM=1024

## Diffusion
diffusion_steps=100

## Optimization
n_attemp_optim=3
nsamples_in_optim_for_mmd=250


# For models
condition_on=1
nfeatures= X.shape[1]

## Train

### Consistency Models

Train conditional model  - P(Y|X=x)

In [ ]:
B, C = X.shape
nfeatures = C - condition_on

# Use the converted data
data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha)

Cos_ConsistencyModeliCT = ConsistencyModeliCT(nfeatures=nfeatures, condition_on=condition_on, nunits=128)
Cos_ConsistencyModeliCT.train_model(
    X=None,
    nepochs=nepochs_CM,
    batch_size=batch_size_CM,
    device=device,
    condition=condition_on,
    data_generator=data_generator,
    use_improved_training=True
)

### Diffusion

Train conditional model P(Y|X=x)

In [ ]:
## LGD
# init a model, train
X = dist_utils.generate_mog_samples(1_000, mu_list, Sigma_list, alpha).float().to(device)
nfeatures = X.shape[1]
condition_on = mu_list[0].shape[0] - mog_means[0].shape[0]
model_cond = Diffusion.DiffusionModel(nfeatures=nfeatures, nblocks=nblocks, nunits=nunits, condition=True,
                                      condition_on=condition_on, diffusion_steps=diffusion_steps)

data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=None)
losses = model_cond.train_model(None,
                                data_generator=data_generator,
                                nepochs=nepochs, batch_size=batch_size, condition_on=condition_on)

Train unconditional model P(X=x)



In [ ]:
# init a model, train
X_for_cond_only = X[:, :model_cond.condition_on]
nfeatures = X_for_cond_only.shape[1]

data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=lambda X: X[:, :model_cond.condition_on])
model_uncond=Diffusion.DiffusionModel(nfeatures=nfeatures, nblocks=nblocks, nunits=nunits, condition=False,diffusion_steps=diffusion_steps)
losses = model_uncond.train_model(None,
                                  data_generator=data_generator,
                                  nepochs=nepochs
, batch_size=batch_size, condition_on=condition_on)


### Flow

In [ ]:
input_dim =mog_means[0].shape[0]
condition_on = mu_list[0].shape[0]-mog_means[0].shape[0]
hidden_dim = nunits
depth = nblocks

Train conditional model P(Y|X=x)



In [ ]:
vf_y_cond_x = FlowMatching.FMModel(nfeatures=input_dim,
                                   condition_on=condition_on,
                                   nunits=hidden_dim,
                                   nblocks=depth,
                                   device=device)

data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=None)
vf_y_cond_x.train_FM(lr=0.001, batch_size=batch_size, nepochs=nepochs,data_generator=data_generator)



Train unconditional model P(X=x)



In [ ]:
input_dim = mu_list[0].shape[0]-mog_means[0].shape[0]#first_column_x.shape[1]
vf_X = FlowMatching.FMModel(nfeatures=input_dim,
                                   condition_on=0,
                                   nunits=hidden_dim,
                                   nblocks=depth,
                                   device=device)
data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=lambda X: X[:, :condition_on])

vf_X.train_FM(lr=0.001, batch_size=batch_size, nepochs=nepochs,
              data_generator=data_generator
              )



## Optimize

In [ ]:
def gmm_l2_distance(mu_p, Sigma_p, w_p, mu_q, Sigma_q, w_q):
    """
    Exact L2 distance between two GMMs:
    ||p - q||^2 = <p,p> - 2<p,q> + <q,q>
    where <f,g> = integral f(x)g(x)dx, closed form for Gaussians.
    """
    import torch

    def gaussian_inner_product(mu1, S1, w1_list, mu2, S2, w2_list):
        # sum_{i,j} w1_i * w2_j * N(mu1_i; mu2_j, S1_i + S2_j)
        total = 0.0
        for mu_i, S_i, w_i in zip(mu1, S1, w1_list):
            for mu_j, S_j, w_j in zip(mu2, S2, w2_list):
                S_sum = S_i + S_j
                diff  = mu_i - mu_j
                d     = mu_i.shape[0]
                sign, logdet = torch.linalg.slogdet(S_sum)
                log_val = -0.5 * (d * torch.log(torch.tensor(2 * 3.14159265)) + logdet
                                  + diff @ torch.linalg.inv(S_sum) @ diff)
                total += w_i.item() * w_j.item() * torch.exp(log_val).item()
        return total

    pp = gaussian_inner_product(mu_p, Sigma_p, w_p, mu_p, Sigma_p, w_p)
    qq = gaussian_inner_product(mu_q, Sigma_q, w_q, mu_q, Sigma_q, w_q)
    pq = gaussian_inner_product(mu_p, Sigma_p, w_p, mu_q, Sigma_q, w_q)
    return pp - 2 * pq + qq

### LGD

In [ ]:
best_x_t_LGD_list = []
l2_gmm_LGD_list   = []
l2_x_LGD_list     = []
lgd_times         = []
final_loss_LGD    = []

for i in trange(n_attemp_optim):
    start_time = time.time()
    best_x_t, best_x_0_cont_xt_hat, final_loss = Optimization.optimize_LGD(
        model_uncond, model_cond, mog_means, mog_variances, weights,
        mu_list, Sigma_list, alpha,
        nsamples=nsamples_in_optim_for_mmd, loss="MMD", device=device)
    end_time = time.time()
    lgd_times.append(end_time - start_time)
    final_loss_LGD.append(final_loss)
    best_x_t_LGD_list.append(best_x_t)

    x_pred_t = best_x_t.float().view(-1).cpu()
    mu_pred, Sigma_pred = dist_utils.compute_conditionals(mu_list, Sigma_list, x_pred_t)
    w_pred = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_pred_t)

    l2_gmm = gmm_l2_distance(mu_pred, Sigma_pred, w_pred, mog_means, mog_variances, weights)
    l2_x   = (x_pred_t - x_star.float().cpu()).pow(2).sum().sqrt().item()

    l2_gmm_LGD_list.append(l2_gmm)
    l2_x_LGD_list.append(l2_x)
    print(f"[{i+1}] L2 GMM: {l2_gmm:.6f}  |  L2 to x*: {l2_x:.6f}")

### LGD-CM

Run the optimization of the LGD-CM

In [ ]:
best_x_t_LGD_CM_list = []
l2_gmm_LGD_CM_list   = []
l2_x_LGD_CM_list     = []
lgd_cm_times         = []
final_loss_LGD_CM    = []

for i in trange(n_attemp_optim):
    start_time = time.time()
    best_x_t, best_x_0_cont_xt_hat, final_loss = Optimization.optimize_LGD(
        model_uncond, Cos_ConsistencyModeliCT, mog_means, mog_variances, weights,
        mu_list, Sigma_list, alpha,
        nsamples=nsamples_in_optim_for_mmd, loss="MMD", device=device,
        CM=True, FLAG=False, num_x_t=3)
    end_time = time.time()
    lgd_cm_times.append(end_time - start_time)
    final_loss_LGD_CM.append(final_loss)
    best_x_t_LGD_CM_list.append(best_x_t)

    x_pred_t = best_x_t.float().view(-1).cpu()
    mu_pred, Sigma_pred = dist_utils.compute_conditionals(mu_list, Sigma_list, x_pred_t)
    w_pred = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_pred_t)

    l2_gmm = gmm_l2_distance(mu_pred, Sigma_pred, w_pred, mog_means, mog_variances, weights)
    l2_x   = (x_pred_t - x_star.float().cpu()).pow(2).sum().sqrt().item()

    l2_gmm_LGD_CM_list.append(l2_gmm)
    l2_x_LGD_CM_list.append(l2_x)
    print(f"[{i+1}] L2 GMM: {l2_gmm:.6f}  |  L2 to x*: {l2_x:.6f}")

### D-FLOW

In [ ]:
x_optim_dflow_list    = []
l2_gmm_dflow_list     = []
l2_x_dflow_list       = []
dflow_times           = []
final_loss_dflow_list = []

for i in trange(n_attemp_optim):
    start_time = time.time()
    x_optim, final_loss = Optimization.optimize_DFLOW(
        vf_y_cond_x, vf_X, device, mog_means, mog_variances, weights,
        max_iter=100, FLAG=False, n_sample=nsamples_in_optim_for_mmd,
        loss_method="MMD", line_search_fn="strong_wolfe")
    end_time = time.time()
    dflow_times.append(end_time - start_time)
    final_loss_dflow_list.append(final_loss)
    x_optim_dflow_list.append(x_optim)

    x_pred_t = x_optim.float().view(-1).cpu()
    mu_pred, Sigma_pred = dist_utils.compute_conditionals(mu_list, Sigma_list, x_pred_t)
    w_pred = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_pred_t)

    l2_gmm = gmm_l2_distance(mu_pred, Sigma_pred, w_pred, mog_means, mog_variances, weights)
    l2_x   = (x_pred_t - x_star.float().cpu()).pow(2).sum().sqrt().item()

    l2_gmm_dflow_list.append(l2_gmm)
    l2_x_dflow_list.append(l2_x)
    print(f"[{i+1}] L2 GMM: {l2_gmm:.6f}  |  L2 to x*: {l2_x:.6f}")


In [ ]:

def summary_row(name, l2_gmm, l2_x, times):
    return {
        "Method":               name,
        "L2 GMM mean":          f"{np.mean(l2_gmm):.4f}",
        "L2 GMM std":           f"{np.std(l2_gmm):.4f}",
        "L2 to x* mean":        f"{np.mean(l2_x):.4f}",
        "L2 to x* std":         f"{np.std(l2_x):.4f}",
        "Time mean (s)":        f"{np.mean(times):.2f}",
        "Time std (s)":         f"{np.std(times):.2f}",
    }

rows = [
    summary_row("LGD",     l2_gmm_LGD_list,    l2_x_LGD_list,    lgd_times),
    summary_row("LGD-CM",  l2_gmm_LGD_CM_list, l2_x_LGD_CM_list, lgd_cm_times),
    summary_row("D-Flow",  l2_gmm_dflow_list,  l2_x_dflow_list,  dflow_times),
]

df = pd.DataFrame(rows).set_index("Method")
display(df)

## Summary - to 10

# 10D cond on 1D

## Parameters for tests

In [ ]:
## NN
nblocks=5
nunits=128
nepochs=30_000
batch_size=512

nepochs_CM=7_500
batch_size_CM=4096

## Diffusion
diffusion_steps=100

## Optimization
n_attemp_optim=25
nsamples_in_optim_for_mmd=250

In [ ]:
mu_list, Sigma_list, alpha,mog_means, mog_variances,weights,x_star= dist_utils.get_param_mog_with_target(dim_data=10,num_components=4,device='cpu',conditional_modes=2,distanceOrScale="Distance")
mog_means, mog_variances, weights= dist_utils.filter_and_normalize(mog_means, mog_variances, weights, threshold=0.001)

In [ ]:
x_star

## Train

### Consistency Models

Train conditional model  - P(Y|X=x)

In [ ]:
# init a model, train
X = dist_utils.generate_mog_samples(1_000, mu_list, Sigma_list, alpha).float().to(device)

condition_on=9
B, C = X.shape
nfeatures = C - condition_on
mu_list = [mu.float() for mu in mu_list]
Sigma_list = [cov.float() for cov in Sigma_list]
alpha = alpha.float()

data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha)

Cos_ConsistencyModeliCT = ConsistencyModeliCT(nfeatures=nfeatures, condition_on=condition_on, nunits=256)
Cos_ConsistencyModeliCT.train_model(X=None, nepochs=nepochs_CM
                      ,batch_size=batch_size_CM, device= device, condition=condition_on,
                      data_generator=data_generator,
                      )

### Diffusion

Train conditional model P(Y|X=x)

In [ ]:
# init a model, train
X = dist_utils.generate_mog_samples(1_000, mu_list, Sigma_list, alpha).float().to(device)
nfeatures = X.shape[1]
condition_on = mu_list[0].shape[0] - mog_means[0].shape[0]
model_cond = Diffusion.DiffusionModel(nfeatures=nfeatures, nblocks=nblocks, nunits=nunits, condition=True,
                                      condition_on=condition_on, diffusion_steps=diffusion_steps)

data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=None)
losses = model_cond.train_model(None,
                                data_generator=data_generator,
                                nepochs=nepochs, batch_size=batch_size, condition_on=condition_on)

Train uncnditional model P(X=x)

In [ ]:
# init a model, train
X_for_cond_only = X[:, :model_cond.condition_on]
nfeatures = X_for_cond_only.shape[1]

data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=lambda X: X[:, :model_cond.condition_on])
model_uncond=Diffusion.DiffusionModel(nfeatures=nfeatures, nblocks=nblocks, nunits=nunits, condition=False,diffusion_steps=diffusion_steps)
losses = model_uncond.train_model(None,
                                  data_generator=data_generator,
                                  nepochs=nepochs
, batch_size=batch_size, condition_on=condition_on)


### Flow

In [ ]:
input_dim =mog_means[0].shape[0]
y_dim = mu_list[0].shape[0]-mog_means[0].shape[0]
hidden_dim = nunits
depth = nblocks


Train conditional model P(Y|X=x)

In [ ]:
vf_y_cond_x = FlowMatching.FMModel(nfeatures=input_dim,
                                   condition_on=condition_on,
                                   nunits=hidden_dim,
                                   nblocks=depth,
                                   device=device)

data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=None)
vf_y_cond_x.train_FM(lr=0.001, batch_size=batch_size, nepochs=nepochs,data_generator=data_generator)



Train conditional model P(X=x)

In [ ]:

input_dim = mu_list[0].shape[0]-mog_means[0].shape[0]#first_column_x.shape[1]
vf_X = FlowMatching.FMModel(nfeatures=input_dim,
                                   condition_on=0,
                                   nunits=hidden_dim,
                                   nblocks=depth,
                                   device=device)
data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=lambda X: X[:, :condition_on])

vf_X.train_FM(lr=0.001, batch_size=batch_size, nepochs=nepochs,
              data_generator=data_generator
              )

## Optimize

### LGD

In [ ]:
best_x_t_LGD_list = []
l2_gmm_LGD_list   = []
l2_x_LGD_list     = []
lgd_times         = []
final_loss_LGD    = []

for i in trange(n_attemp_optim):
    start_time = time.time()
    best_x_t, best_x_0_cont_xt_hat, final_loss = Optimization.optimize_LGD(
        model_uncond, model_cond, mog_means, mog_variances, weights,
        mu_list, Sigma_list, alpha,
        nsamples=nsamples_in_optim_for_mmd, loss="MMD", device=device)
    best_x_t = best_x_t.reshape(-1, 1)
    end_time = time.time()
    lgd_times.append(end_time - start_time)
    final_loss_LGD.append(final_loss)
    best_x_t_LGD_list.append(best_x_t)

    x_pred_t = best_x_t.float().view(-1).cpu()
    mu_pred, Sigma_pred = dist_utils.compute_conditionals(mu_list, Sigma_list, x_pred_t)
    w_pred = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_pred_t)

    l2_gmm = gmm_l2_distance(mu_pred, Sigma_pred, w_pred, mog_means, mog_variances, weights)
    l2_x   = (x_pred_t - x_star.float().cpu()).pow(2).sum().sqrt().item()

    l2_gmm_LGD_list.append(l2_gmm)
    l2_x_LGD_list.append(l2_x)
    print(f"[{i+1}] L2 GMM: {l2_gmm:.6f}  |  L2 to x*: {l2_x:.6f}")


### LGD-CM

In [ ]:
best_x_t_LGD_CM_list = []
l2_gmm_LGD_CM_list   = []
l2_x_LGD_CM_list     = []
lgd_cm_times         = []
final_loss_LGD_CM    = []

for i in trange(n_attemp_optim):
    start_time = time.time()
    best_x_t, best_x_0_cont_xt_hat, final_loss = Optimization.optimize_LGD(
        model_uncond, Cos_ConsistencyModeliCT, mog_means, mog_variances, weights,
        mu_list, Sigma_list, alpha,
        nsamples=nsamples_in_optim_for_mmd, loss="MMD", device=device,
        CM=True, FLAG=False, num_x_t=10)
    best_x_t = best_x_t.reshape(-1, 1)
    end_time = time.time()
    lgd_cm_times.append(end_time - start_time)
    final_loss_LGD_CM.append(final_loss)
    best_x_t_LGD_CM_list.append(best_x_t)

    x_pred_t = best_x_t.float().view(-1).cpu()
    mu_pred, Sigma_pred = dist_utils.compute_conditionals(mu_list, Sigma_list, x_pred_t)
    w_pred = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_pred_t)

    l2_gmm = gmm_l2_distance(mu_pred, Sigma_pred, w_pred, mog_means, mog_variances, weights)
    l2_x   = (x_pred_t - x_star.float().cpu()).pow(2).sum().sqrt().item()

    l2_gmm_LGD_CM_list.append(l2_gmm)
    l2_x_LGD_CM_list.append(l2_x)
    print(f"[{i+1}] L2 GMM: {l2_gmm:.6f}  |  L2 to x*: {l2_x:.6f}")


### D-FLOW

In [ ]:
x_optim_dflow_list    = []
l2_gmm_dflow_list     = []
l2_x_dflow_list       = []
dflow_times           = []
final_loss_dflow_list = []

for i in trange(n_attemp_optim):
    start_time = time.time()
    x_optim, final_loss = Optimization.optimize_DFLOW(
        vf_y_cond_x, vf_X, device, mog_means, mog_variances, weights,
        max_iter=100, FLAG=False, n_sample=nsamples_in_optim_for_mmd,
        loss_method="MMD", line_search_fn="strong_wolfe")
    x_optim = x_optim.reshape(-1, 1)
    end_time = time.time()
    dflow_times.append(end_time - start_time)
    final_loss_dflow_list.append(final_loss)
    x_optim_dflow_list.append(x_optim)

    x_pred_t = x_optim.float().view(-1).cpu()
    mu_pred, Sigma_pred = dist_utils.compute_conditionals(mu_list, Sigma_list, x_pred_t)
    w_pred = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_pred_t)

    l2_gmm = gmm_l2_distance(mu_pred, Sigma_pred, w_pred, mog_means, mog_variances, weights)
    l2_x   = (x_pred_t - x_star.float().cpu()).pow(2).sum().sqrt().item()

    l2_gmm_dflow_list.append(l2_gmm)
    l2_x_dflow_list.append(l2_x)
    print(f"[{i+1}] L2 GMM: {l2_gmm:.6f}  |  L2 to x*: {l2_x:.6f}")



In [ ]:
rows = [
    summary_row("LGD",     l2_gmm_LGD_list,    l2_x_LGD_list,    lgd_times),
    summary_row("LGD-CM",  l2_gmm_LGD_CM_list, l2_x_LGD_CM_list, lgd_cm_times),
    summary_row("D-Flow",  l2_gmm_dflow_list,  l2_x_dflow_list,  dflow_times),
]

df = pd.DataFrame(rows).set_index("Method")
display(df)